# Campus Help Desk — Agentic RAG Demo

A simple **agentic workflow** built with **LangGraph** + **RAG** (Retrieval-Augmented Generation).

**The idea:** a student asks a question, a *router* decides how to handle it, and the answer is grounded in a small campus knowledge base.

```
            +-------------------+
  query --> |      router       |
            +---------+---------+
                      |
        +-------------+--------------+----------------+
        v             v              v
   greeting_node   rag_node      fun_fact_node
   (template)   (retrieve+LLM)   (curated list)
```

**Why this is a good teaching example:**
- The `rag_node` answers from *your* FAQs, not the model's memory.
- The router shows how an agent picks *different tools* for different questions.
- A distance threshold lets the agent say **"I don't know"** instead of making things up.

## 1. Setup

These packages are already installed in the course environment. If you are running somewhere new (e.g. Colab), uncomment the pip line below.

We use a small, fast, instruction-tuned model (`flan-t5-small`, ~300 MB) so this runs in minutes with **no API key**.

In [ ]:
# !pip install -q faiss-cpu sentence-transformers transformers langgraph

import random
import numpy as np
import faiss
import torch
from typing import TypedDict
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langgraph.graph import StateGraph, END

print("Imports OK")

Imports OK


## 2. Knowledge base + retriever (the "R" in RAG)

We embed a handful of campus FAQs and store them in a FAISS vector index. `retrieve()` returns the closest facts **with their distances** so we can judge how relevant they are.

In [ ]:
campus_faqs = [
    "Library hours: The main library is open 8am-10pm on weekdays and 10am-6pm on weekends.",
    "Wi-Fi: Connect to the 'Campus-Secure' network using your student ID and password. Guests use 'Campus-Guest'.",
    "Registration: Course registration opens two weeks before each semester through the student portal.",
    "Parking: Students can buy a parking permit online. Lot C is free after 5pm.",
    "Dining: The main dining hall serves meals 7am-9pm and accepts meal plan swipes and card payments.",
    "Recreation: The gym and pool are open 6am-11pm and are free for enrolled students with an ID card.",
    "Advising: Academic advisors are available by appointment through the advising portal, Monday to Friday.",
    "IT Help Desk: For tech issues, visit the help desk in the library lobby or email support@campus.edu.",
    "Exams: Final exam schedules are posted on the registrar's website four weeks before finals week.",
    "Clubs: There are over 100 student clubs. Visit the Student Life office to join or start a club.",
]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embed_model.encode(campus_faqs)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.array(kb_embeddings))
print(f"Indexed {index.ntotal} campus FAQs.")

In [ ]:
# Lower distance = more similar. Tune this threshold to control when the agent says "I don't know".
DISTANCE_THRESHOLD = 1.3

def retrieve(query, top_k=2):
    query_emb = embed_model.encode([query])
    distances, indices = index.search(np.array(query_emb), top_k)
    return [(campus_faqs[i], float(d)) for d, i in zip(distances[0], indices[0])]

# Quick sanity check
retrieve("When is the library open?")

## 3. The generator (the "G" in RAG)

A small instruction-tuned model turns the retrieved context into a friendly answer. The first run downloads the model (~300 MB).

In [ ]:
# flan-t5-small is a small instruction-tuned seq2seq model (~300 MB on first run).
# We load it directly because the transformers 5.x pipeline no longer exposes a
# "text2text-generation" task.
MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def call_llm(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        output_ids = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print(call_llm("Greet a new student in one friendly sentence."))

## 4. Define the state and the nodes

Each node is just a function that reads the shared `state` and writes to it. Notice the three different *kinds* of node:
- `greeting_node` — a simple template (no model needed)
- `fun_fact_node` — picks from a curated list
- `rag_node` — retrieves facts **and** generates a grounded answer (with an "I don't know" fallback)

In [ ]:
class HelpDeskState(TypedDict):
    query: str
    answer: str
    selected_node: str

FUN_FACTS = [
    "The library hosts a 'therapy dog' day every exam week to help students de-stress.",
    "The oldest building on campus is over 120 years old and has a hidden study room.",
    "The student-run radio station has been broadcasting for more than 40 years.",
]

def router_node(state: HelpDeskState):
    q = state["query"].lower()
    if any(w in q for w in ["hi", "hello", "hey", "thanks", "thank you"]):
        state["selected_node"] = "greeting_node"
    elif "fun fact" in q or "fun" in q:
        state["selected_node"] = "fun_fact_node"
    else:
        state["selected_node"] = "rag_node"
    return state

def rag_node(state: HelpDeskState):
    print("RAG node activated \U0001F4DA")
    results = retrieve(state["query"], top_k=2)
    best_text, best_dist = results[0]
    if best_dist > DISTANCE_THRESHOLD:
        state["answer"] = (
            "I'm not sure about that one. Try rephrasing, or contact the "
            "Student Life office for more help."
        )
        return state
    context = "\n".join(text for text, _ in results)
    prompt = (
        "You are a helpful campus assistant. Use the context to answer the question.\n"
        f"Context:\n{context}\n\nQuestion: {state['query']}\nAnswer:"
    )
    state["answer"] = call_llm(prompt)
    return state

def greeting_node(state: HelpDeskState):
    print("Greeting node activated \U0001F44B")
    state["answer"] = (
        "Hi there! I'm the campus help desk assistant. Ask me about the library, "
        "Wi-Fi, parking, dining, advising, and more!"
    )
    return state

def fun_fact_node(state: HelpDeskState):
    print("Fun fact node activated \u2728")
    state["answer"] = random.choice(FUN_FACTS)
    return state

## 5. Build the LangGraph workflow

The router is the entry point. Its output (`selected_node`) decides which branch runs via conditional edges.

In [ ]:
workflow = StateGraph(HelpDeskState)
workflow.add_node("router", router_node)
workflow.add_node("rag_node", rag_node)
workflow.add_node("greeting_node", greeting_node)
workflow.add_node("fun_fact_node", fun_fact_node)

workflow.set_entry_point("router")
workflow.add_conditional_edges(
    "router",
    lambda s: s["selected_node"],
    {
        "rag_node": "rag_node",
        "greeting_node": "greeting_node",
        "fun_fact_node": "fun_fact_node",
    },
)
workflow.add_edge("rag_node", END)
workflow.add_edge("greeting_node", END)
workflow.add_edge("fun_fact_node", END)

app = workflow.compile()
print("Graph compiled \u2705")

In [ ]:
# Dependency-free diagram of the graph (paste into any Mermaid viewer to visualize).
print(app.get_graph().draw_mermaid())

## 6. Test it

Watch which node activates for each query. The last question is intentionally off-topic to trigger the fallback.

In [ ]:
queries = [
    "Hello!",
    "When is the library open on weekends?",
    "How do I connect to the Wi-Fi?",
    "Tell me a fun fact about campus.",
    "Where can I park for free?",
    "What is the capital of France?",
]

for q in queries:
    result = app.invoke({"query": q, "answer": "", "selected_node": ""})
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print("-" * 60)